# 00 — Baseline univariado: pH da estação EF01 (CETESB)

**Objetivo:** estabelecer os baselines que qualquer modelo futuro precisa bater na série de pH (5 min, 01/06–31/08/2026).
**Decisões travadas:** variável `pH` · lookback `L=2016` (7 dias) · horizonte `H=12` (1 h) · split temporal 70/15/15 sem shuffle · baselines clássicos (persistência, sazonal-naive, média móvel, ARIMA, Prophet).
**Dados:** `dados/ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md` (encoding Windows-1252, `;`, vírgula decimal, ~18% de faltantes, validados até 22/08/2026).

In [1]:
import pickle
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

# --- caminhos (funciona com cwd = repo ou notebooks/) ---
ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- hiperparâmetros do experimento ---
L, H = 2016, 12          # lookback (7 dias) e horizonte (1 h) em passos de 5 min
SEASON = 288              # ciclo diário em passos de 5 min
INTERP_LIMIT = 24         # interpolação temporal máx. (24 passos = 2 h); gaps maiores viram NaN
PROVISORIO_CORTE = "2026-08-22 09:00"  # dados após isso não são validados (ver dados/README.md)
ARIMA_ORDER = (2, 1, 2)
ARIMA_STRIDE = 96         # ARIMA é reestimado a cada 96 origens de teste (custo); demais modelos usam todas

print("ROOT:", ROOT, "| CSV existe:", CSV.exists())

ROOT: /home/marcos/Projetos/temporal-model | CSV existe: True


## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(26209, 2) 2026-06-01 00:00:00 → 2026-08-31 00:00:00
faltantes: 4794 (18.3%)


,ds,y
count,26209,21415.000000
mean,2026-07-16 12:00:00,6.142466
min,2026-06-01 00:00:00,5.560000
25%,2026-06-23 18:00:00,5.970000
50%,2026-07-16 12:00:00,6.170000
75%,2026-08-08 06:00:00,6.280000
max,2026-08-31 00:00:00,6.590000
std,NaN,0.198893


## 2. EDA — perfil, faltantes e ciclo diário

In [3]:
# maior gap consecutivo (em passos de 5 min)
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvline(pd.Timestamp(PROVISORIO_CORTE), color="r", ls="--", lw=1)
ax[0].set_title("pH EF01 — série completa (vermelho = início do trecho provisório)")
ax[0].set_ylabel("pH")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do pH")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("pH por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")

maior gap: 18 passos = 1.5 h | gaps > 24 passos: 0


fig salva: /home/marcos/Projetos/temporal-model/resultados/figs/01-eda.png


## 3. Limpeza — grade completa + interpolação limitada
Reindex na grade de 5 min, flag do trecho provisório e interpolação temporal de no máximo 2 h. Gaps maiores permanecem `NaN` e as janelas que os contêm são descartadas (sem vazamento).

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)} (iguais = nenhum timestamp ausente)")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
provisorio = s.index > pd.Timestamp(PROVISORIO_CORTE)
print(f"trecho provisório: {int(provisorio.sum())} slots ({100*provisorio.mean():.1f}%)")

# overlay cru x preenchido numa semana de exemplo
amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 26209 | linhas no CSV: 26209 (iguais = nenhum timestamp ausente)
NaN após interpolação (limite 24): 0
trecho provisório: 2484 slots (9.5%)


fig salva


## 4. Estacionariedade (ADF) e decomposição STL
STL roda nos últimos 4032 pontos do treino (rápido e representativo); tendência/sazonalidade/resíduo orientam a escolha de `lookback`.

In [5]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária (usar diferenciação / modelos robustos)'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-4.66 p-valor=0.000102 → estacionária


fig salva


## 5. Janelamento e split temporal
Amostras `(L=2016 → H=12)` por janela deslizante; só janelas 100% observadas (pós-interpolação); split 70/15/15 **sem shuffle**.

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)          # (n_janelas, L+H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]             # timestamp do último alvo de cada janela
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)
splits = {"train": (0, i1), "val": (i1, i2), "test": (i2, n)}
for k, (a, b) in splits.items():
    print(f"{k}: {b-a} janelas | alvos {ends[a].date()} → {ends[b-1].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")

train: 16927 janelas | alvos 2026-06-08 → 2026-08-05
val: 3627 janelas | alvos 2026-08-05 → 2026-08-18
test: 3628 janelas | alvos 2026-08-18 → 2026-08-31
janelas descartadas (com NaN): 0


## 6. Baselines baratos (todas as origens de teste)
Persistência, sazonal-naive (lag 288 = 1 dia) e média móvel das últimas 288 obs.

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

a, b = splits["test"]
Xte, Yte = X[a:b], Y[a:b]
pred = {
    "persistencia": np.repeat(Xte[:, -1:], H, axis=1),
    "sazonal_naive_288": np.stack([Xte[:, L - SEASON + h] for h in range(H)], axis=1),
    "media_movel_288": np.repeat(Xte[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
}
res = pd.DataFrame({m: metricas(Yte, p) for m, p in pred.items()}).T.round(4)
print(res.to_string())
res.to_csv(OUT / "metricas_baseline.csv")
print("salvo:", OUT / "metricas_baseline.csv")

                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0368  0.0542  0.5942  0.5942
sazonal_naive_288  0.0470  0.0654  0.7578  0.7577
media_movel_288    0.0618  0.0773  0.9943  0.9944
salvo: /home/marcos/Projetos/temporal-model/resultados/metricas_baseline.csv


## 7. ARIMA (subamostra de origens — custo)
Reestimado a cada 96 origens de teste sobre a própria janela de entrada (`ARIMA(2,1,2)`); falhas isoladas caem para persistência. O subconjunto é o mesmo usado na tabela comparável da §9.

In [8]:
import time

idx_arima = np.arange(0, len(Xte), ARIMA_STRIDE)
Ya, Xa = Yte[idx_arima], Xte[idx_arima]
Pa = np.empty_like(Ya)
t0 = time.time()
for i, (x, y0) in enumerate(zip(Xa, Ya)):
    try:
        Pa[i] = ARIMA(x, order=ARIMA_ORDER).fit().get_forecast(H).predicted_mean.values
    except Exception:
        Pa[i] = np.repeat(x[-1], H)   # fallback honesto: persistência
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(Xa)} origens...", flush=True)
print(f"ARIMA: {len(Xa)} origens em {time.time()-t0:.0f}s |", metricas(Ya, Pa))

# artefato: ARIMA ajustado na cauda do treino (para inspeção/reuso)
with open(OUT / "modelos" / "arima212_cauda_treino.pkl", "wb") as f:
    pickle.dump(ARIMA(s.iloc[n_train-4032:n_train].interpolate().values, order=ARIMA_ORDER).fit(), f)
print("modelo salvo")

  10/38 origens...


  20/38 origens...


  30/38 origens...


ARIMA: 38 origens em 81s | {'MAE': 0.029780701754386008, 'RMSE': 0.043908547303637704, 'MAPE': 0.4817487066813309, 'sMAPE': 0.4809524878927961}


modelo salvo


## 8. Prophet (opcional — pula se `prophet`/CmdStan indisponível)
Um único ajuste no treino (tolera `NaN`) e projeção sobre o teste; métricas nas mesmas janelas.

In [9]:
PROPHET_OK = False
try:
    from prophet import Prophet
    import cmdstanpy
    assert cmdstanpy.cmdstan_path() is not None
    df_train = pd.DataFrame({"ds": s.index[:n_train], "y": s.values[:n_train]}).dropna()
    m = Prophet(daily_seasonality=True, weekly_seasonality=True)
    m.fit(df_train)
    fut = m.predict(pd.DataFrame({"ds": s.index}))
    # reconstrução direta: para a origem i, alvos = ends[a+i] - H + 1 ... ends[a+i]
    fmap = fut.set_index("ds")["yhat"]
    Pp = np.stack([[fmap.loc[d - pd.Timedelta(minutes=5*(H-1-h))] for h in range(H)] for d in ends[a:b]])
    PROPHET_OK = True
    print("Prophet:", metricas(Yte, Pp))
    from prophet.serialize import model_to_json
    (OUT / "modelos" / "prophet_ph.json").write_text(model_to_json(m))
    print("modelo salvo")
except Exception as e:
    print(f"Prophet pulado ({type(e).__name__}: {str(e)[:150]}). Rode `uv pip install prophet` + CmdStan e reexecute.")

Importing plotly failed. Interactive plots will not work.


21:14:19 - cmdstanpy - INFO - Chain [1] start processing


21:14:40 - cmdstanpy - INFO - Chain [1] done processing


Prophet: {'MAE': 0.04709771774108913, 'RMSE': 0.062098979781021024, 'MAPE': 0.7576567255424055, 'sMAPE': 0.7579113653275743}
modelo salvo


## 9. Comparação final + figuras
Tabela cheia (modelos baratos + Prophet nas origens de teste) e tabela no subconjunto ARIMA (comparação justa). Baseline a bater = melhor MAE.

In [10]:
linhas = {m: metricas(Yte, p) for m, p in pred.items()}
if PROPHET_OK:
    linhas["prophet"] = metricas(Yte, Pp)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste completo ===")
print(tab.to_string())

# subconjunto ARIMA (justo)
sub = {m: metricas(Ya, p[idx_arima]) for m, p in pred.items()}
sub["arima_212"] = metricas(Ya, Pa)
if PROPHET_OK:
    sub["prophet"] = metricas(Ya, Pp[idx_arima])
tab_sub = pd.DataFrame(sub).T.round(4)
print("=== subconjunto ARIMA ===")
print(tab_sub.to_string())
print(f"\nBaseline a bater (menor MAE no teste): {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")

=== teste completo ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0368  0.0542  0.5942  0.5942
sazonal_naive_288  0.0470  0.0654  0.7578  0.7577
media_movel_288    0.0618  0.0773  0.9943  0.9944
prophet            0.0471  0.0621  0.7577  0.7579
=== subconjunto ARIMA ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0298  0.0439  0.4817  0.4810
sazonal_naive_288  0.0463  0.0668  0.7461  0.7465
media_movel_288    0.0581  0.0703  0.9379  0.9369
arima_212          0.0298  0.0439  0.4817  0.4810
prophet            0.0484  0.0626  0.7799  0.7800

Baseline a bater (menor MAE no teste): persistencia = 0.0368


In [11]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    t = pd.date_range(ends[a:b][k] - pd.Timedelta(minutes=5*(L-1)), ends[a:b][k], freq="5min")
    ax.plot(t, Xte[k], lw=0.8, label="contexto (cauda)")
    tf = pd.date_range(ends[a:b][k] - pd.Timedelta(minutes=5*(H-1)), ends[a:b][k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred["persistencia"][k], ":", lw=1, label="persistência")
    ax.set_title(f"origem {ends[a:b][k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste — baselines (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")
print("figs salvas")

figs salvas


## 10. Conclusões e próximos passos

- O **baseline a bater** (menor MAE, tabela §9) é a régua do projeto: LSTM/GRU (§7 do README) e depois TFT/PatchTST (§3) precisam superá-lo **no mesmo split**.
- ARIMA foi avaliado em subamostra por custo — o número honesto para comparar é a tabela do subconjunto.
- Se o Prophet foi pulado, instale o CmdStan (`uv pip install prophet` já feito; completar toolchain C++) e reexecute só a §8.
- Artefatos em `resultados/`: `metricas_baseline.csv`, `modelos/` (ARIMA + Prophet) e `figs/`.